<img src="../../../assets/images/logos/ucu_logo_clean.svg" alt="UCU Logo" width="200" style="float: right; margin: 0 0 10px 10px;"/>

### Matemáticas para Aprendizaje Automático - 2026

--------
## Laboratorio 2.2: Descomposición en Valores Singulares (SVD)

#### Objetivos

- Calcular la SVD por dos caminos distintos: diagonalizando $A^\top A$ y diagonalizando la matriz simétrica aumentada.
- Verificar cada implementación contra `numpy.linalg.svd` en reconstrucción, ortogonalidad y valores singulares.
- Aplicar la aproximación de rango bajo a la compresión de imágenes y cuantificar el compromiso entre rango y error.
- Construir un sistema de recomendación por filtrado colaborativo completando una matriz con datos faltantes.
- Calcular la pseudoinversa de Moore-Penrose a partir de la SVD y comprobar sus cuatro condiciones.

## Introducción

La descomposición en valores singulares escribe cualquier matriz, cuadrada o no,
como $A = U\Sigma V^\top$ con $U$ y $V$ ortogonales y $\Sigma$ diagonal no
negativa. A diferencia de la diagonalización, que exige una matriz cuadrada y
diagonalizable, la SVD existe siempre. Eso la convierte en la herramienta por
defecto para trabajar con matrices de datos, que casi nunca son cuadradas.

Su utilidad viene de una propiedad concreta: truncar la suma
$A = \sum_i \sigma_i u_i v_i^\top$ a los primeros $k$ términos da la mejor
aproximación de rango $k$ que existe, medida en norma de Frobenius. De ahí salen
la compresión de imágenes, el análisis de componentes principales, el filtrado
colaborativo de los sistemas de recomendación y la pseudoinversa que resuelve
sistemas sin solución única. En este laboratorio se recorren esas cuatro
aplicaciones después de construir la SVD desde cero por dos caminos distintos.

In [ ]:
# Librerías necesarias
import io
import os
import zipfile

import numpy as np
import scipy.linalg as la
import matplotlib.pyplot as plt
import matplotlib.cbook as cbook
from matplotlib.image import imread
import pandas as pd
import requests

## Parte 1: SVD mediante descomposición de autovalores

Dada una matriz $A \in \mathbb{R}^{m \times n}$, la SVD es la factorización

$$A = U \Sigma V^{\top}$$

donde $U \in \mathbb{R}^{m \times m}$ y $V \in \mathbb{R}^{n \times n}$ son
ortogonales, y $\Sigma \in \mathbb{R}^{m \times n}$ es diagonal con entradas no
negativas $\sigma_1 \geq \sigma_2 \geq \cdots \geq 0$, los **valores
singulares**.

#### Paso 1: los vectores singulares derechos

Al formar el producto $A^\top A$ aparece la diagonalización buscada:

$$A^{\top} A = (U\Sigma V^{\top})^{\top} (U\Sigma V^{\top}) = V \Sigma^{\top} U^{\top} U \Sigma V^{\top} = V (\Sigma^{\top} \Sigma) V^{\top}$$

Como $U^\top U = I$, esto dice que:

- los **autovectores** de $A^\top A$ son las columnas de $V$, los vectores singulares derechos;
- los **autovalores** de $A^\top A$ son $\lambda_i = \sigma_i^2$.

De la misma forma, $A A^{\top} = U (\Sigma \Sigma^{\top}) U^{\top}$ entrega las
columnas de $U$, los vectores singulares izquierdos, y los mismos autovalores.

> **Criterio práctico:** si $m \geq n$ conviene diagonalizar $A^\top A$, que es
> $n \times n$; si $m < n$, diagonalizar $A A^\top$, que es $m \times m$. Así se
> trabaja siempre con la matriz más chica.

#### Paso 2: los valores singulares

$$\sigma_i = \sqrt{\lambda_i}, \qquad \lambda_i \geq 0$$

Los autovalores de $A^\top A$ son no negativos porque la matriz es semidefinida
positiva. En punto flotante conviene escribir
$\sigma_i = \sqrt{\max(\lambda_i, 0)}$ para no tomar la raíz de un número
negativo generado por redondeo.

#### Paso 3: los vectores singulares que faltan

Con $V$ y $\Sigma$ conocidos, la relación $A = U\Sigma V^\top$ da las columnas
de $U$ una por una:

$$A v_i = \sigma_i u_i \quad \Longrightarrow \quad u_i = \frac{1}{\sigma_i} A v_i \qquad (\sigma_i > 0)$$

Si se parte de $U$, la relación simétrica es
$v_i = \frac{1}{\sigma_i} A^\top u_i$. Las columnas asociadas a valores
singulares nulos no quedan determinadas por esta fórmula y hay que completarlas
con cualquier base ortonormal del complemento, por ejemplo con Gram-Schmidt.

### Ejercicio L2.2.1: SVD por diagonalización de $A^\top A$

Implementá `svd_via_eig(A)`, que devuelve `U`, `S` y `Vt` con la misma
convención que `np.linalg.svd(A, full_matrices=True)`.

**a)** Diagonalizá $A^\top A$ con `np.linalg.eigh` y ordená autovalores y
autovectores en orden descendente.

**b)** Obtené los valores singulares como raíz de los autovalores, recortando
los negativos por redondeo.

**c)** Construí las columnas de $U$ con $u_i = A v_i / \sigma_i$ y completá las
$m - n$ columnas restantes con Gram-Schmidt.

**Nota:** `np.linalg.eigh` está pensada para matrices simétricas y devuelve los
autovalores en orden ascendente, así que hay que invertir el orden. Los signos
de los vectores singulares no son únicos: $(u_i, v_i)$ y $(-u_i, -v_i)$ dan la
misma matriz, y por eso la verificación compara valores singulares y
reconstrucción, no las columnas una a una.

In [ ]:
def svd_via_eig(A, seed=0):
    """
    Calcula la SVD diagonalizando A^T A.

    Args:
        A: Matriz de tamaño m x n
        seed: Semilla para completar las columnas de U asociadas a sigma = 0

    Returns:
        U: Matriz ortogonal m x m
        S: Vector de valores singulares en orden descendente
        Vt: Traspuesta de la matriz ortogonal V (n x n)
    """
    m, n = A.shape
    rng = np.random.default_rng(seed)

    # a) diagonalizar A^T A y ordenar en forma descendente
    eigenvalues, V = np.linalg.eigh(A.T @ A)
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[idx]
    V = V[:, idx]

    # b) valores singulares
    S = np.sqrt(np.clip(eigenvalues, 0, None))

    # c) vectores singulares izquierdos: u_i = A v_i / sigma_i
    U = np.zeros((m, m))
    for i in range(min(m, n)):
        if S[i] > 1e-10:
            U[:, i] = A @ V[:, i] / S[i]

    # completar las columnas restantes con Gram-Schmidt
    for i in range(m):
        if np.linalg.norm(U[:, i]) > 1e-10:
            continue
        v = rng.standard_normal(m)
        for j in range(m):
            if j != i and np.linalg.norm(U[:, j]) > 1e-10:
                v -= np.dot(v, U[:, j]) * U[:, j]
        U[:, i] = v / np.linalg.norm(v)

    Vt = V.T
    return U, S, Vt


# ── Verificación ─────────────────────────────────────────────────────────────────
rng = np.random.default_rng(42)
A = rng.random((4, 3))
m, n = A.shape

U_custom, S_custom, Vt_custom = svd_via_eig(A)
Sigma_custom = np.zeros((m, n))
np.fill_diagonal(Sigma_custom, S_custom)
A_reconstruida = U_custom @ Sigma_custom @ Vt_custom

U_numpy, S_numpy, Vt_numpy = np.linalg.svd(A, full_matrices=True)

error_reconstruccion = np.linalg.norm(A - A_reconstruida, 'fro') / np.linalg.norm(A, 'fro')
error_ortogonalidad_U = np.linalg.norm(U_custom.T @ U_custom - np.eye(m), 'fro')
error_ortogonalidad_V = np.linalg.norm(Vt_custom @ Vt_custom.T - np.eye(n), 'fro')

print("Valores singulares propios :", np.round(S_custom, 8))
print("Valores singulares de NumPy:", np.round(S_numpy, 8))
print(f"Error relativo de reconstrucción: {error_reconstruccion:.2e}")
print(f"Error de ortogonalidad de U     : {error_ortogonalidad_U:.2e}")
print(f"Error de ortogonalidad de V     : {error_ortogonalidad_V:.2e}")

assert error_reconstruccion < 1e-10, "U S Vt no reconstruye A"
assert error_ortogonalidad_U < 1e-10, "U no es ortogonal"
assert error_ortogonalidad_V < 1e-10, "V no es ortogonal"
assert np.allclose(S_custom, S_numpy), "Los valores singulares difieren de los de NumPy"
assert np.all(np.diff(S_custom) <= 1e-12), "Los valores singulares deben quedar en orden descendente"

## Parte 2 (OPCIONAL): SVD mediante la matriz simétrica aumentada

> **Nota:** esta parte es opcional / bonus. Profundiza en la estabilidad numérica del cálculo de la SVD, pero no es necesaria para el resto del laboratorio ni para el Laboratorio 2.3.

Hay un segundo camino para llegar a la SVD que evita formar $A^\top A$. Se
define la matriz simétrica de tamaño $(m+n) \times (m+n)$

$$H = \begin{bmatrix} 0 & A \\ A^{\top} & 0 \end{bmatrix}$$

Si $A = U\Sigma V^\top$ tiene rango $r$, se comprueba que para cada
$i = 1, \dots, r$ el vector

$$q_i^{\pm} = \frac{1}{\sqrt{2}} \begin{bmatrix} u_i \\ \pm v_i \end{bmatrix}$$

cumple $H q_i^{\pm} = \pm\sigma_i\, q_i^{\pm}$. Es decir, el espectro de $H$ es
$\{\pm\sigma_1, \dots, \pm\sigma_r\}$ junto con $m + n - 2r$ autovalores nulos, y
cada autovector apila un vector singular izquierdo con uno derecho. Recuperar la
SVD consiste entonces en quedarse con los autovalores positivos y separar cada
autovector en sus dos mitades, multiplicando por $\sqrt{2}$ para renormalizar.

Por qué importa: $A^\top A$ eleva al cuadrado el número de condición
($\kappa(A^\top A) = \kappa(A)^2$), así que los valores singulares chicos se
pierden en el redondeo. La matriz $H$ es simétrica y tiene los $\sigma_i$ como
autovalores directamente, sin elevar al cuadrado. Este es el camino que discuten
Trefethen y Bau en *Numerical Linear Algebra*, Lecture 31.

Este desarrollo se trabaja en el problema 2.2.6 de la hoja de ejercicios 2 de la
unidad 2, que conviene resolver antes de encarar el ejercicio siguiente.

### Ejercicio L2.2.2 (OPCIONAL): SVD por la matriz aumentada

*Ejercicio opcional / bonus — no obligatorio para la entrega.*

Implementá `svd_via_hermitian(A)` con la misma firma que `svd_via_eig`.

**a)** Construí $H$ y diagonalizala con `np.linalg.eigh`.

**b)** Contá cuántos autovalores superan la tolerancia, quedate con esos en
orden descendente y separá cada autovector en las mitades $u_i$ y $v_i$.

**c)** Completá $U$ y $V$ hasta ser cuadradas usando `np.linalg.qr` sobre
$[\,U_r \mid I\,]$.

**d)** Compará el resultado con el del Ejercicio L2.2.1 y con NumPy.

**Nota:** `eigh` devuelve los autovalores en orden ascendente, de modo que los
positivos quedan al final del arreglo. Al completar con QR hay que corregir los
signos con `np.sign(np.diag(R))` para que las primeras $r$ columnas queden
iguales a $U_r$ y no a $\pm U_r$, porque el signo de $u_i$ tiene que seguir
emparejado con el de $v_i$.

In [ ]:
# OPCIONAL (bonus) — no obligatorio para la entrega
def svd_via_hermitian(A, tol=1e-10):
    """
    Calcula la SVD diagonalizando la matriz simétrica aumentada
    H = [[0, A], [A^T, 0]].

    Args:
        A: Matriz de tamaño m x n
        tol: Umbral por debajo del cual un autovalor se considera nulo

    Returns:
        U: Matriz ortogonal m x m
        S: Vector con los r valores singulares positivos, en orden descendente
        Vt: Traspuesta de la matriz ortogonal V (n x n)
    """
    m, n = A.shape

    # a) construir y diagonalizar H
    H = np.zeros((m + n, m + n))
    H[:m, m:] = A
    H[m:, :m] = A.T
    eigenvalues, eigenvectors = np.linalg.eigh(H)

    # b) autovalores positivos en orden descendente y separación de los autovectores
    r = int(np.sum(eigenvalues > tol))
    orden = np.argsort(eigenvalues)[::-1][:r]
    S = eigenvalues[orden]
    Q_pos = eigenvectors[:, orden]
    U_r = np.sqrt(2) * Q_pos[:m, :]
    V_r = np.sqrt(2) * Q_pos[m:, :]

    # c) completar U y V hasta ser cuadradas, conservando los signos de U_r y V_r
    def completar(base, dim):
        if base.shape[1] == dim:
            return base
        Q, R = np.linalg.qr(np.hstack([base, np.eye(dim)]))
        signos = np.sign(np.diag(R[:base.shape[1], :base.shape[1]]))
        signos[signos == 0] = 1
        Q[:, :base.shape[1]] *= signos
        return Q[:, :dim]

    U = completar(U_r, m)
    V = completar(V_r, n)
    return U, S, V.T


# ── Verificación ─────────────────────────────────────────────────────────────────
U_h, S_h, Vt_h = svd_via_hermitian(A)
r = len(S_h)

Sigma_h = np.zeros((m, n))
np.fill_diagonal(Sigma_h, S_h)
A_rec_h = U_h @ Sigma_h @ Vt_h

print("Valores singulares por H     :", np.round(S_h, 8))
print("Valores singulares por A^T A :", np.round(S_custom[:r], 8))
print("Valores singulares de NumPy  :", np.round(S_numpy[:r], 8))
print(f"Rango detectado: r = {r}")
print(f"Error relativo de reconstrucción: {np.linalg.norm(A - A_rec_h, 'fro') / np.linalg.norm(A, 'fro'):.2e}")

assert np.allclose(S_h, S_numpy[:r]), "Los valores singulares difieren de los de NumPy"
assert np.allclose(S_h, S_custom[:r]), "Los dos métodos propios no coinciden"
assert np.linalg.norm(A - A_rec_h, 'fro') / np.linalg.norm(A, 'fro') < 1e-10, \
    "La reconstrucción por la matriz aumentada no recupera A"
assert np.linalg.norm(U_h.T @ U_h - np.eye(m), 'fro') < 1e-10, "U no es ortogonal"
assert np.linalg.norm(Vt_h @ Vt_h.T - np.eye(n), 'fro') < 1e-10, "V no es ortogonal"

## Parte 3: Compresión de imágenes

Una imagen en escala de grises es una matriz $A \in \mathbb{R}^{m \times n}$ de
intensidades. Escribiendo su SVD como suma de matrices de rango 1,

$$A = U \Sigma V^{\top} = \sum_{i=1}^{r} \sigma_i \, u_i v_i^{\top}$$

cada término aporta un modo de la imagen, y los $\sigma_i$ ordenados de mayor a
menor dicen cuánta energía aporta cada uno.

#### Aproximación de rango $k$ (teorema de Eckart y Young)

La mejor aproximación de rango $k$ en norma de Frobenius se obtiene truncando la
suma:

$$A_k = \sum_{i=1}^{k} \sigma_i \, u_i v_i^{\top} = U_k \Sigma_k V_k^{\top}$$

y el error queda determinado por los valores singulares descartados:

$$\|A - A_k\|_F = \sqrt{\sigma_{k+1}^2 + \cdots + \sigma_r^2}$$

O sea que la calidad de la compresión depende de con qué velocidad decae el
espectro. Una imagen con estructura (bordes alineados, zonas planas, texturas
repetidas) tiene un decaimiento rápido; ruido blanco tiene un espectro plano y
no se comprime.

#### Ahorro de almacenamiento

| Representación | Números a almacenar |
|---|---|
| Imagen original $A$ | $m \times n$ |
| SVD truncada de rango $k$ | $k(m + n + 1)$ |

La compresión conviene cuando $k(m+n+1) \ll mn$, es decir cuando
$k \ll \frac{mn}{m+n}$.

### Ejercicio L2.2.3: Compresión por rango bajo

**a)** Implementá `compress_matrix(A, k)`, que devuelve la aproximación de rango
$k$ de $A$.

**b)** Completá `display_compressed_images` para recorrer varios rangos y mostrar
la imagen reconstruida junto con la fracción de almacenamiento
$k(m+n+1)/(mn)$.

**c)** OPCIONAL: probá con una imagen propia cambiando `image_path` y observá
cómo cambia el rango necesario según la cantidad de detalle.

**Nota:** usá `np.linalg.svd(A, full_matrices=False)`, que devuelve solo las
primeras $\min(m,n)$ columnas y alcanza para truncar. La imagen de prueba es una
de las que trae matplotlib, así que el ejercicio corre sin descargar nada.

In [ ]:
def compress_matrix(A, k):
    """
    Comprime una matriz A quedándose con los k primeros valores singulares.

    Args:
        A: Matriz de tamaño m x n
        k: Rango de la aproximación

    Returns:
        A_compressed: Matriz de rango k más cercana a A en norma de Frobenius
    """
    A = A.astype(np.float64)

    U, S, Vt = np.linalg.svd(A, full_matrices=False)
    U_k = U[:, :k]
    S_k = S[:k]
    Vt_k = Vt[:k, :]
    A_compressed = U_k @ np.diag(S_k) @ Vt_k
    return A_compressed


def cargar_imagen_gris(image_path=None):
    """
    Carga una imagen en escala de grises como matriz de float64.
    Si no se pasa una ruta, usa una imagen de muestra de matplotlib.
    """
    if image_path is None:
        with cbook.get_sample_data('grace_hopper.jpg') as fh:
            im = imread(fh)
    else:
        im = imread(image_path)
    img = im[:, :, 0] if im.ndim == 3 else im
    return img.astype(np.float64)


def display_compressed_images(img, ranks=(1, 2, 5, 10, 20, 50)):
    """
    Muestra la imagen comprimida con distintos rangos y devuelve las fracciones
    de almacenamiento correspondientes.
    """
    m, n = img.shape
    _, singular_values, _ = np.linalg.svd(img, full_matrices=False)

    compressed_images = []
    compression_ratios = []

    for k in ranks:
        im_k = compress_matrix(img, k)
        ratio = k * (m + n + 1) / (m * n)
        compressed_images.append(im_k)
        compression_ratios.append(ratio)

    n_images = len(ranks) + 1
    fig, axes = plt.subplots(2, (n_images + 2) // 2, figsize=(15, 8))
    axes = axes.flatten()

    axes[0].imshow(img, cmap='gray')
    axes[0].set_title('Original')
    axes[0].axis('off')

    for i, (im_k, ratio, k) in enumerate(zip(compressed_images, compression_ratios, ranks)):
        axes[i + 1].imshow(im_k, cmap='gray')
        axes[i + 1].set_title(f'k={k}, ratio={ratio:.2f}')
        axes[i + 1].axis('off')

    for ax in axes[n_images:]:
        ax.axis('off')
    if len(axes) > n_images:
        ax = axes[-1]
        ax.axis('on')
        ax.semilogy(singular_values[:100])
        ax.set_title('Valores singulares')
        ax.set_xlabel('Índice')
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    return compression_ratios


# ── Verificación ─────────────────────────────────────────────────────────────────
img = cargar_imagen_gris()
ranks = (1, 2, 5, 10, 20, 50)
ratios = display_compressed_images(img, ranks=ranks)

S_img = np.linalg.svd(img, compute_uv=False)
for k in ranks:
    A_k = compress_matrix(img, k)
    assert np.linalg.matrix_rank(A_k) <= k, f"La aproximación de rango {k} tiene rango mayor"
    # Eckart-Young: el error de Frobenius es exactamente la cola del espectro
    error_real = np.linalg.norm(img - A_k, 'fro')
    error_teorico = np.sqrt(np.sum(S_img[k:] ** 2))
    assert np.isclose(error_real, error_teorico, rtol=1e-8), \
        f"El error para k={k} no coincide con la cota de Eckart-Young"

assert len(ratios) == len(ranks), "Falta calcular alguna fracción de almacenamiento"
assert all(r2 > r1 for r1, r2 in zip(ratios, ratios[1:])), \
    "La fracción de almacenamiento debería crecer con k"
print("\nEl error de cada truncamiento coincide con sqrt(sum_{i>k} sigma_i^2), como predice Eckart-Young.")

### Ejercicio L2.2.4: Análisis cuantitativo de la compresión

Más allá de mirar las imágenes, conviene medir. Definí la **energía acumulada**
hasta el rango $k$ como

$$E(k) = \frac{\sum_{i=1}^{k} \sigma_i^2}{\sum_{i=1}^{r} \sigma_i^2}$$

que es la fracción de $\|A\|_F^2$ que retiene la aproximación de rango $k$. El
error relativo correspondiente sale de Eckart y Young sin necesidad de
reconstruir la matriz:

$$\frac{\|A - A_k\|_F}{\|A\|_F} = \frac{\sqrt{\sum_{i>k} \sigma_i^2}}{\|\sigma\|_2} = \sqrt{1 - E(k)}$$

**a)** Calculá `energia` y `error_vs_k` a partir del vector de valores
singulares, sin reconstruir ninguna matriz.

**b)** Graficá el espectro, el error relativo y la energía acumulada.

**c) (OPCIONAL)** Armá la tabla con el rango $k$ necesario para retener 80, 90, 95 y 99 por
ciento de la energía, con su fracción de almacenamiento.

**Nota:** `np.cumsum` y `np.argmax` sobre un arreglo booleano resuelven casi
todo. `np.argmax(energia >= 0.9)` devuelve el primer índice donde se alcanza el
umbral.

In [ ]:
m_img, n_img = img.shape
S_img = np.linalg.svd(img, compute_uv=False)
r_img = len(S_img)

energia = np.cumsum(S_img ** 2) / np.sum(S_img ** 2)
k_range = np.arange(1, min(r_img, 300) + 1)
norma_total = np.linalg.norm(S_img)
cola_cuadrados = norma_total ** 2 - np.cumsum(S_img ** 2)[k_range - 1]
error_vs_k = np.sqrt(np.clip(cola_cuadrados, 0, None)) / norma_total

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].semilogy(S_img[:150], 'b-')
axes[0].set_title('Espectro de valores singulares')
axes[0].set_xlabel('Índice i')
axes[0].set_ylabel('sigma_i (escala log)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(k_range, error_vs_k, 'r-')
axes[1].set_xlabel('Rango k')
axes[1].set_ylabel('Error relativo de Frobenius')
axes[1].set_title('||A - A_k||_F / ||A||_F  frente a k')
axes[1].grid(True, alpha=0.3)

axes[2].plot(k_range, energia[:len(k_range)] * 100, 'g-')
for umbral, color, ls in [(90, 'k', '--'), (95, 'b', '-.'), (99, 'r', ':')]:
    k_u = int(np.argmax(energia >= umbral / 100)) + 1
    ratio_u = k_u * (m_img + n_img + 1) / (m_img * n_img)
    axes[2].axhline(umbral, color=color, linestyle=ls,
                    label=f'{umbral}%  (k={k_u}, ratio={ratio_u:.2f})')
axes[2].set_xlabel('Rango k')
axes[2].set_ylabel('Energía acumulada (%)')
axes[2].set_title('Energía acumulada frente a k')
axes[2].legend(fontsize=8)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# OPCIONAL (parte c): tabla de k necesario por umbral de energía
print(f"\nImagen: {m_img} x {n_img} = {m_img * n_img:,} píxeles   |   rango efectivo: {r_img}")
print(f"{'Umbral':>10}  {'k':>6}  {'Almac. SVD':>12}  {'Almac. orig.':>13}  {'Ratio':>7}")
for umbral in [80, 90, 95, 99]:
    k = int(np.argmax(energia >= umbral / 100)) + 1
    almac = k * (m_img + n_img + 1)
    print(f"{umbral:>9}%  {k:>6,}  {almac:>12,}  {m_img * n_img:>13,}  {almac / (m_img * n_img):>7.3f}")

# ── Verificación ─────────────────────────────────────────────────────────────────
assert np.isclose(energia[-1], 1.0), "La energía acumulada debe llegar a 1"
assert np.all(np.diff(energia) >= -1e-12), "La energía acumulada debe ser no decreciente"
assert np.all(np.diff(error_vs_k) <= 1e-12), "El error debe decrecer al aumentar k"
# la identidad error(k) = sqrt(1 - E(k)) vincula ambas curvas
assert np.allclose(error_vs_k, np.sqrt(np.clip(1 - energia[:len(k_range)], 0, None)), atol=1e-8), \
    "No se cumple la relación error(k) = sqrt(1 - E(k))"
# el error medido sobre la matriz reconstruida coincide con el predicho
k_check = 20
error_medido = np.linalg.norm(img - compress_matrix(img, k_check), 'fro') / np.linalg.norm(img, 'fro')
assert np.isclose(error_medido, error_vs_k[k_check - 1], rtol=1e-8), \
    "El error predicho no coincide con el medido sobre la reconstrucción"

## Parte 4 (OPCIONAL, pero muy recomendada): Sistema de recomendación

> **Nota:** esta parte es opcional para la entrega, pero es muy importante que la hagan: es el acercamiento más cercano a un problema real de estos algoritmos, aplicando SVD sobre datos reales (MovieLens) para armar un sistema de recomendación, con las imperfecciones típicas de los datos reales (faltantes, ruido) incluidas.

Un sistema de recomendación sugiere productos a partir del comportamiento
pasado de los usuarios. El caso clásico es el de una plataforma de streaming:
se arma una matriz de calificaciones $C \in \mathbb{R}^{m \times n}$ donde cada
fila es un usuario, cada columna una película y $C_{ij}$ la nota que el usuario
$i$ le puso a la película $j$.

El problema es que casi ninguna entrada está observada. En el conjunto de datos
que se usa en esta parte, menos del 2 por ciento de las celdas tiene valor; el
resto es `NaN`. Recomendar consiste en estimar esas entradas faltantes.

#### La hipótesis de rango bajo

El **filtrado colaborativo** se apoya en una suposición sobre la estructura de
$C$: aunque la matriz sea grande, unos pocos factores explican la mayor parte de
la variabilidad. Si existen $k$ perfiles de gusto (comedia liviana, cine de
autor, acción, y así), entonces

$$C \approx U_k \Sigma_k V_k^{\top}$$

donde la fila $i$ de $U_k\Sigma_k$ describe cuánto pesa cada factor en el usuario
$i$, y la columna $j$ de $V_k^\top$ describe cuánto carga la película $j$ en cada
factor. La calificación predicha es el producto interno de esos dos vectores.

#### Cómo aplicar SVD con datos faltantes

La SVD no acepta `NaN`, así que hace falta imputar primero. La estrategia más
simple es rellenar cada faltante con la media de su columna, o sea suponer que
un usuario cualquiera le pondría a esa película su nota promedio. Después se
calcula la SVD de rango $k$ de la matriz completada y se lee la predicción en las
posiciones que estaban vacías.

La imputación por la media es una suposición fuerte y sesga la solución hacia el
promedio. Una mejora usual es iterar: recalcular la SVD reemplazando los
faltantes por la predicción anterior en lugar de por la media, y repetir hasta
que la matriz deje de cambiar.

#### Elección de $k$

El valor de $k$ controla el compromiso de siempre. Con $k$ chico el modelo no
captura la estructura; con $k$ grande empieza a ajustar el ruido de la
imputación. Se elige ocultando parte de las calificaciones conocidas, prediciendo
y midiendo la raíz del error cuadrático medio sobre lo oculto:

$$\text{RMSE} = \sqrt{\frac{1}{|T|} \sum_{(i,j) \in T} (\hat{C}_{ij} - C_{ij})^2}$$

### Ejercicio L2.2.5 (OPCIONAL): Completar una matriz con datos faltantes

Implementá `completar_matriz(matriz, k)`, que recibe una matriz con `NaN` y
devuelve una matriz completa estimada con SVD de rango $k$.

**a)** Reemplazá los `NaN` por la media de cada columna. Si una columna entera
es `NaN`, es decir una película sin ninguna calificación, usá la media global.

**b)** Calculá la SVD de la matriz ya completada y truncala a $k$ factores.

**c)** OPCIONAL: implementá la versión iterativa. En cada paso, en lugar de la
media de columna, usá como relleno la predicción de la iteración anterior, y
cortá cuando el cambio entre iteraciones sea menor a una tolerancia.

**Nota:** `np.nanmean` calcula medias ignorando `NaN`, pero avisa si una columna
entera es `NaN`, así que conviene tratar ese caso antes. `np.isnan` da la
máscara de faltantes, que hace falta guardar para saber dónde leer las
predicciones.

In [ ]:
# OPCIONAL — ver nota en la Parte 4
def completar_matriz(matriz, k, n_iter=1, tol=1e-4):
    """
    Completa una matriz con valores NaN utilizando SVD de rango k.

    Args:
        matriz: Matriz de calificaciones con NaN en las entradas no observadas
        k: Número de factores latentes
        n_iter: Cantidad de iteraciones (1 equivale a imputar solo por la media)
        tol: Tolerancia de corte para la versión iterativa

    Returns:
        matriz_completada: Matriz reconstruida, sin NaN
    """
    nan_mask = np.isnan(matriz)

    # a) imputación inicial por la media de cada columna
    col_means = ...     # COMPLETAR: media de cada columna ignorando los NaN
    global_mean = ...   # COMPLETAR: media global, para columnas enteramente NaN
    col_means = np.where(np.isnan(col_means), global_mean, col_means)
    actual = ...        # COMPLETAR: matriz con los NaN reemplazados

    # b) y c) SVD truncada, iterando si se pide
    for _ in range(n_iter):
        U, S, Vt = ...            # COMPLETAR: SVD reducida de la matriz actual
        matriz_completada = ...   # COMPLETAR: reconstrucción con k factores
        siguiente = np.where(nan_mask, matriz_completada, matriz)
        cambio = np.linalg.norm(siguiente - actual) / max(np.linalg.norm(actual), 1e-12)
        actual = siguiente
        if cambio < tol:
            break

    return matriz_completada


# ── Verificación ─────────────────────────────────────────────────────────────────
# Se construye una matriz de rango exactamente 3 y se le borran entradas al azar:
# si la hipótesis de rango bajo vale, la reconstrucción debe recuperarlas bien.
rng_rec = np.random.default_rng(7)
rango_real = 3
M_true = rng_rec.random((60, 40)) @ rng_rec.random((40, rango_real)) @ rng_rec.random((rango_real, 40))
M_obs = M_true.copy()
faltantes = rng_rec.choice(M_obs.size, size=400, replace=False)
M_obs.ravel()[faltantes] = np.nan
mask_faltantes = np.isnan(M_obs)

M_completada = completar_matriz(M_obs, k=rango_real, n_iter=30)
error_faltantes = (np.linalg.norm(M_completada[mask_faltantes] - M_true[mask_faltantes])
                   / np.linalg.norm(M_true[mask_faltantes]))

print(f"Entradas ocultadas: {mask_faltantes.sum()} de {M_obs.size}")
print(f"Error relativo sobre las entradas ocultadas: {error_faltantes:.3e}")
print(f"¿Quedan NaN en la matriz completada? {np.any(np.isnan(M_completada))}")

assert not np.any(np.isnan(M_completada)), "La matriz completada no debería tener NaN"
assert M_completada.shape == M_obs.shape, "La forma de la matriz completada cambió"
assert np.linalg.matrix_rank(M_completada) <= rango_real + 1e-9, "La reconstrucción no es de rango k"
assert error_faltantes < 0.05, "La reconstrucción no recupera las entradas ocultadas"

### Ejercicio L2.2.6 (OPCIONAL): Recomendador de películas sobre MovieLens

Armá un recomendador sobre el conjunto MovieLens (versión *latest-small*, unas
100000 calificaciones de 610 usuarios sobre 9700 películas).

Se proveen ya resueltas las funciones de carga y armado de la matriz:

- `load_movie_data`: descarga el conjunto una sola vez, lo cachea en `data/` y devuelve dos `DataFrame`, uno de calificaciones y otro de películas.
- `create_user_item_matrix`: arma la matriz usuario por ítem con `NaN` en las entradas no observadas, junto con los diccionarios que mapean identificadores a índices.
- `predict_ratings`: envoltorio de `completar_matriz`.

Se pide:

**a)** Cargar los datos, armar la matriz y verificar que las predicciones no
tienen `NaN` y caen en un rango de calificaciones razonable.

**b)** Completar `evaluate_recommender`: ocultar el 20 por ciento de las
calificaciones conocidas de cada usuario, predecir con distintos valores de $k$,
calcular el RMSE sobre lo oculto y graficarlo. Elegir el mejor $k$.

**c)** Completar `recommend_movies`: dado un usuario, predecir sus
calificaciones, descartar las películas que ya calificó y devolver las
`top_n` de mayor nota predicha.

**d)** Para tres usuarios, identificar su película favorita ya calificada,
borrarla de la matriz, recalcular las recomendaciones y ver si la película
vuelve a aparecer entre las primeras.

**Nota:** la parte **d)** es la validación más honesta del recomendador: se le
esconde una respuesta que ya se conoce y se mira si la reconstruye. Ojo con no
recomendar películas que el usuario ya calificó, que es el error clásico en este
ejercicio.

In [ ]:
# OPCIONAL — ver nota en la Parte 4
def load_movie_data(data_dir="data"):
    """
    Carga MovieLens (versión latest-small). Si no está en disco, lo descarga.

    Returns:
        ratings_df, movies_df
    """
    zip_url = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
    os.makedirs(data_dir, exist_ok=True)
    ratings_file = os.path.join(data_dir, "ratings.csv")
    movies_file = os.path.join(data_dir, "movies.csv")

    if not os.path.exists(ratings_file) or not os.path.exists(movies_file):
        print("Descargando datos de MovieLens...")
        try:
            response = requests.get(zip_url, timeout=60)
            response.raise_for_status()
        except Exception as e:
            raise RuntimeError(
                "No se pudieron descargar los datos de MovieLens.\n"
                f"  Motivo: {type(e).__name__}: {e}\n\n"
                "Descargalos a mano y volvé a ejecutar esta celda:\n"
                f"  1. Bajá {zip_url}\n"
                "  2. Descomprimí el zip.\n"
                f"  3. Copiá ratings.csv y movies.csv a la carpeta '{data_dir}/'.\n\n"
                "Nota: si el error menciona el certificado del servidor, el problema es del sitio "
                "de MovieLens y no de tu instalación. No deshabilites la verificación TLS."
            ) from e
        with zipfile.ZipFile(io.BytesIO(response.content)) as z:
            with z.open("ml-latest-small/ratings.csv") as f:
                pd.read_csv(f).to_csv(ratings_file, index=False)
            with z.open("ml-latest-small/movies.csv") as f:
                pd.read_csv(f).to_csv(movies_file, index=False)
        print("Descarga completada.")

    return pd.read_csv(ratings_file), pd.read_csv(movies_file)


def create_user_item_matrix(ratings_df):
    """Arma la matriz usuario por ítem, con NaN donde no hay calificación."""
    user_ids = ratings_df['userId'].unique()
    movie_ids = ratings_df['movieId'].unique()
    user_mapper = {uid: idx for idx, uid in enumerate(user_ids)}
    item_mapper = {mid: idx for idx, mid in enumerate(movie_ids)}
    matriz = np.full((len(user_ids), len(movie_ids)), np.nan)
    filas = ratings_df['userId'].map(user_mapper).to_numpy()
    columnas = ratings_df['movieId'].map(item_mapper).to_numpy()
    matriz[filas, columnas] = ratings_df['rating'].to_numpy()
    return matriz, user_mapper, item_mapper


def predict_ratings(user_item_matrix, k=20):
    """Completa la matriz de calificaciones con SVD de rango k."""
    return completar_matriz(user_item_matrix, k)


def evaluate_recommender(user_item_matrix, k_values=(5, 10, 20, 50, 100), seed=0):
    """
    Oculta el 20% de las calificaciones de cada usuario, predice con distintos k
    y devuelve el RMSE sobre los datos ocultados.
    """
    rng_eval = np.random.default_rng(seed)
    matriz_entrenamiento = np.copy(user_item_matrix)
    mascara_prueba = np.zeros(user_item_matrix.shape, dtype=bool)

    for i in range(user_item_matrix.shape[0]):
        calificados = np.where(~np.isnan(user_item_matrix[i]))[0]
        if len(calificados) > 0:
            n_test = max(1, int(0.2 * len(calificados)))
            indices_test = rng_eval.choice(calificados, n_test, replace=False)
            matriz_entrenamiento[i, indices_test] = np.nan
            mascara_prueba[i, indices_test] = True

    datos_ocultos = user_item_matrix[mascara_prueba]

    resultados = []
    for k in k_values:
        predicciones = ...   # COMPLETAR: predecir con rango k y leer la máscara de prueba
        rmse = ...           # COMPLETAR: RMSE entre predicciones y datos_ocultos
        resultados.append((k, rmse))
        print(f"k={k:>4}, RMSE={rmse:.4f}")

    k_vals, rmse_vals = zip(*resultados)
    plt.figure(figsize=(8, 5))
    plt.plot(k_vals, rmse_vals, 'b-o')
    plt.xlabel('Número de factores latentes (k)')
    plt.ylabel('RMSE')
    plt.title('Evaluación del recomendador SVD')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    return dict(resultados)


def recommend_movies(user_id, user_item_matrix, user_mapper, item_mapper, movies_df,
                     k=50, top_n=10, verbose=True):
    """
    Devuelve las top_n películas recomendadas para un usuario, excluyendo las
    que ya calificó.

    Returns:
        Lista de tuplas (movieId, título, calificación predicha)
    """
    if user_id not in user_mapper:
        raise KeyError(f"El usuario {user_id} no existe en el conjunto de datos")

    user_idx = user_mapper[user_id]
    reverse_item_mapper = {idx: mid for mid, idx in item_mapper.items()}
    predicted_ratings = predict_ratings(user_item_matrix, k)

    puntajes = ...        # COMPLETAR: fila de predicciones del usuario
    ya_calificadas = ...  # COMPLETAR: índices de películas que el usuario ya calificó
    pass                  # COMPLETAR: descartar las ya calificadas de los puntajes
    top_indices = ...     # COMPLETAR: índices de las top_n mejores predicciones

    recomendaciones = []
    for idx in top_indices:
        movie_id = reverse_item_mapper[idx]
        info = movies_df[movies_df['movieId'] == movie_id]
        titulo = info.iloc[0]['title'] if not info.empty else f"(id {movie_id})"
        recomendaciones.append((movie_id, titulo, predicted_ratings[user_idx, idx]))

    if verbose:
        reales = user_item_matrix[user_idx].copy()
        reales[np.isnan(reales)] = -1
        print(f"\nTop 5 ya calificadas por el usuario {user_id}:")
        for i, idx in enumerate(np.argsort(reales)[::-1][:5], 1):
            movie_id = reverse_item_mapper[idx]
            info = movies_df[movies_df['movieId'] == movie_id]
            if not info.empty:
                print(f"  {i}. {info.iloc[0]['title']} (calificación {user_item_matrix[user_idx, idx]:.1f})")
        print(f"\nTop {top_n} recomendaciones para el usuario {user_id}:")
        for i, (_, titulo, puntaje) in enumerate(recomendaciones, 1):
            print(f"  {i}. {titulo} (predicha {puntaje:.2f})")

    return recomendaciones


# ── Verificación ─────────────────────────────────────────────────────────────────
# a) cargar los datos y verificar las predicciones
ratings_df, movies_df = load_movie_data()
user_item_matrix, user_mapper, item_mapper = create_user_item_matrix(ratings_df)

n_conocidas = np.count_nonzero(~np.isnan(user_item_matrix))
print(f"Matriz usuario por ítem: {user_item_matrix.shape}")
print(f"Calificaciones conocidas: {n_conocidas:,} ({n_conocidas / user_item_matrix.size:.2%} de densidad)")

predicted = predict_ratings(user_item_matrix, k=50)
print(f"Rango de calificaciones predichas: [{predicted.min():.2f}, {predicted.max():.2f}]")

assert not np.any(np.isnan(predicted)), "Las predicciones no deberían tener NaN"
assert predicted.min() > -2 and predicted.max() < 8, \
    "Las calificaciones predichas quedaron fuera de un rango razonable"

# b) evaluar y elegir k
print("\n--- Evaluación del recomendador ---")
rmse_results = evaluate_recommender(user_item_matrix, k_values=(5, 10, 20, 50, 100))
best_k = min(rmse_results, key=rmse_results.get)
print(f"\nMejor k según RMSE: {best_k} (RMSE = {rmse_results[best_k]:.4f})")

assert len(rmse_results) == 5, "Falta evaluar algún valor de k"
assert all(0 < v < 5 for v in rmse_results.values()), "Algún RMSE quedó fuera de rango"

# c) recomendar para tres usuarios
for uid in [1, 2, 5]:
    recomendaciones = recommend_movies(uid, user_item_matrix, user_mapper, item_mapper,
                                       movies_df, k=best_k)
    assert len(recomendaciones) == 10, "Deberían devolverse 10 recomendaciones"
    idx_usuario = user_mapper[uid]
    ya_vistas = {list(item_mapper.keys())[list(item_mapper.values()).index(i)]
                 for i in np.where(~np.isnan(user_item_matrix[idx_usuario]))[0]}
    assert not any(mid in ya_vistas for mid, _, _ in recomendaciones), \
        "No hay que recomendar películas que el usuario ya calificó"

In [ ]:
# d) esconder la película favorita y ver si el recomendador la recupera
reverse_item_mapper = {idx: mid for mid, idx in item_mapper.items()}
posiciones_encontradas = []

for test_uid in [1, 2, 5]:
    user_idx = user_mapper[test_uid]

    fav_idx = ...      # COMPLETAR: índice de la película mejor calificada por el usuario
    fav_mid = reverse_item_mapper[fav_idx]
    fav_titulo = movies_df[movies_df['movieId'] == fav_mid].iloc[0]['title']
    fav_rating = user_item_matrix[user_idx, fav_idx]

    print(f"\nUsuario {test_uid}, favorita: '{fav_titulo}' (calificación {fav_rating:.1f})")

    matriz_test = ...  # COMPLETAR: copia de la matriz con la favorita puesta en NaN

    pred = predict_ratings(matriz_test, k=best_k)
    puntajes = pred[user_idx].copy()
    puntajes[np.where(~np.isnan(matriz_test[user_idx]))[0]] = -np.inf

    top20 = np.argsort(puntajes)[::-1][:20]
    top20_mids = [reverse_item_mapper[i] for i in top20]

    if fav_mid in top20_mids:
        posicion = top20_mids.index(fav_mid) + 1
        posiciones_encontradas.append(posicion)
        print(f"  Recuperada en el top 20, posición {posicion}")
    else:
        posiciones_encontradas.append(None)
        print("  No aparece en el top 20")

    print("  Top 5 recomendadas:")
    for rank, idx in enumerate(top20[:5], 1):
        mid = reverse_item_mapper[idx]
        titulo = movies_df[movies_df['movieId'] == mid].iloc[0]['title']
        print(f"    {rank}. {titulo} (predicha {pred[user_idx, idx]:.2f})")

# ── Verificación ─────────────────────────────────────────────────────────────────
assert len(posiciones_encontradas) == 3, "Falta evaluar alguno de los tres usuarios"
print(f"\nPelículas favoritas recuperadas en el top 20: "
      f"{sum(p is not None for p in posiciones_encontradas)} de 3")

## Parte 5: Pseudoinversa mediante SVD

La pseudoinversa de Moore y Penrose, $A^{+}$, extiende la idea de inversa a
matrices rectangulares o singulares, donde $A^{-1}$ no existe. Sirve para
"resolver" $Ax = b$ cuando el sistema no tiene solución exacta (sobredeterminado,
$m > n$) o cuando tiene infinitas (subdeterminado, $m < n$).

#### Definición

$A^{+} \in \mathbb{R}^{n \times m}$ es la única matriz que satisface las cuatro
condiciones:

| # | Condición | Lectura |
|---|---|---|
| 1 | $A A^{+} A = A$ | $A^{+}$ es inversa débil de $A$ |
| 2 | $A^{+} A A^{+} = A^{+}$ | $A$ es inversa débil de $A^{+}$ |
| 3 | $(A A^{+})^{\top} = A A^{+}$ | $A A^{+}$ es simétrica |
| 4 | $(A^{+} A)^{\top} = A^{+} A$ | $A^{+} A$ es simétrica |

Si $A$ es cuadrada e invertible, $A^{+} = A^{-1}$ y las cuatro condiciones se
reducen a $A A^{-1} = I$.

#### Cálculo a partir de la SVD

Dada $A = U\Sigma V^{\top}$, la pseudoinversa se obtiene intercambiando los roles
de $U$ y $V$ e invirtiendo los valores singulares no nulos:

$$A^{+} = V \Sigma^{+} U^{\top}, \qquad
\Sigma^{+}_{ii} = \begin{cases} 1/\sigma_i & \sigma_i > 0 \\ 0 & \sigma_i = 0 \end{cases}$$

En punto flotante no hay valores singulares exactamente nulos, así que se fija un
umbral `tol` y se anula todo $\sigma_i \leq \texttt{tol}$. Ese umbral importa:
invertir un $\sigma_i$ diminuto multiplica el ruido por $1/\sigma_i$.

Que esta fórmula cumple las condiciones se verifica directamente. Por ejemplo,
para la primera:

$$A A^{+} A = (U\Sigma V^{\top})(V\Sigma^{+} U^{\top})(U\Sigma V^{\top}) = U(\Sigma \Sigma^{+} \Sigma)V^{\top} = U\Sigma V^{\top} = A$$

porque cada entrada diagonal cumple $\sigma_i \cdot \frac{1}{\sigma_i} \cdot \sigma_i = \sigma_i$.

#### Qué resuelve

- **Sobredeterminado** ($m > n$): $x^{*} = A^{+}b$ minimiza $\|Ax - b\|_2$, la solución de mínimos cuadrados.
- **Subdeterminado** ($m < n$): $x^{*} = A^{+}b$ es la solución de norma $\|x\|_2$ mínima entre todas las que cumplen $Ax = b$.
- **Cuadrado e invertible**: $x^{*} = A^{+}b = A^{-1}b$.

Estos tres casos son el punto de partida del Laboratorio 2.3.

### Ejercicio L2.2.7: Pseudoinversa por SVD

Implementá `pseudoinverse_svd(A, tol)` usando la fórmula
$A^{+} = V\Sigma^{+}U^{\top}$.

**a)** Calculá la SVD reducida y construí $\Sigma^{+}$ invirtiendo solo los
valores singulares mayores que `tol`.

**b)** Verificá las cuatro condiciones de Moore y Penrose sobre una matriz
rectangular.

**c)** Comprobá que sobre una matriz cuadrada e invertible el resultado coincide
con $A^{-1}$.

**Nota:** `np.where(S > tol, 1.0 / S, 0.0)` genera una advertencia de división
por cero porque evalúa las dos ramas. Para evitarla, calculá los recíprocos solo
donde corresponde.

In [ ]:
def pseudoinverse_svd(A, tol=1e-10):
    """
    Calcula la pseudoinversa de Moore-Penrose mediante SVD.

    Args:
        A: Matriz de tamaño m x n
        tol: Umbral por debajo del cual un valor singular se considera nulo

    Returns:
        A_pinv: Pseudoinversa de A, de tamaño n x m
    """
    U, S, Vt = ...   # COMPLETAR: SVD reducida de A
    S_pinv = ...     # COMPLETAR: recíprocos de los valores singulares mayores que tol, 0 el resto
    A_pinv = ...     # COMPLETAR: V @ diag(S_pinv) @ U.T
    return A_pinv


# ── Verificación ─────────────────────────────────────────────────────────────────
rng_pinv = np.random.default_rng(3)
A_rect = rng_pinv.random((4, 3))
A_pinv = pseudoinverse_svd(A_rect)

err1 = np.linalg.norm(A_rect @ A_pinv @ A_rect - A_rect, 'fro')
err2 = np.linalg.norm(A_pinv @ A_rect @ A_pinv - A_pinv, 'fro')
P = A_rect @ A_pinv
err3 = np.linalg.norm(P.T - P, 'fro')
Q_mat = A_pinv @ A_rect
err4 = np.linalg.norm(Q_mat.T - Q_mat, 'fro')

print(f"Cond 1  ||A A+ A - A||_F   = {err1:.2e}")
print(f"Cond 2  ||A+ A A+ - A+||_F = {err2:.2e}")
print(f"Cond 3  ||(A A+)^T - A A+||_F = {err3:.2e}")
print(f"Cond 4  ||(A+ A)^T - A+ A||_F = {err4:.2e}")

for i, err in enumerate([err1, err2, err3, err4], 1):
    assert err < 1e-10, f"No se cumple la condición {i} de Moore-Penrose"

A_pinv_numpy = np.linalg.pinv(A_rect)
diferencia = np.linalg.norm(A_pinv - A_pinv_numpy, 'fro') / np.linalg.norm(A_pinv_numpy, 'fro')
print(f"\nDiferencia relativa con np.linalg.pinv: {diferencia:.2e}")
assert diferencia < 1e-10, "El resultado difiere de np.linalg.pinv"

# c) matriz cuadrada e invertible: A+ = A^-1
A_sq = rng_pinv.random((5, 5))
err_inv = (np.linalg.norm(pseudoinverse_svd(A_sq) - np.linalg.inv(A_sq), 'fro')
           / np.linalg.norm(np.linalg.inv(A_sq), 'fro'))
err_id = np.linalg.norm(A_sq @ pseudoinverse_svd(A_sq) - np.eye(5), 'fro')
print(f"\nMatriz cuadrada invertible:")
print(f"  ||A+ - A^-1||_F / ||A^-1||_F = {err_inv:.2e}")
print(f"  ||A A+ - I||_F               = {err_id:.2e}")
assert err_inv < 1e-10, "Sobre una matriz invertible A+ debería coincidir con A^-1"
assert err_id < 1e-10, "A A+ debería dar la identidad"

## Conclusiones

En este laboratorio hemos explorado:

1. **SVD por diagonalización de $A^\top A$**: el camino directo, que reduce el
   problema a un problema de autovalores simétrico. Su punto débil es que
   $\kappa(A^\top A) = \kappa(A)^2$, así que los valores singulares chicos se
   degradan.
2. **SVD por la matriz simétrica aumentada**: al diagonalizar
   $H = \left[\begin{smallmatrix} 0 & A \\ A^\top & 0\end{smallmatrix}\right]$
   los valores singulares aparecen como autovalores, sin elevar al cuadrado el
   condicionamiento. Los dos métodos coinciden con `numpy.linalg.svd` hasta
   precisión de máquina.
3. **Compresión de imágenes**: truncar la SVD a rango $k$ da la mejor
   aproximación posible en norma de Frobenius, y el error se lee directamente de
   los valores singulares descartados, sin reconstruir la matriz. Cuánto se puede
   comprimir depende de la velocidad de decaimiento del espectro.
4. **Sistema de recomendación**: la misma idea de rango bajo, aplicada a una
   matriz con más del 98 por ciento de entradas faltantes, permite estimar
   calificaciones no observadas. El valor de $k$ se elige por validación sobre
   datos ocultados, no a ojo.
5. **Pseudoinversa**: $A^{+} = V\Sigma^{+}U^{\top}$ cumple las cuatro condiciones
   de Moore y Penrose y unifica los casos sobredeterminado, subdeterminado e
   invertible. El umbral con que se decide qué valor singular es nulo es una
   decisión numérica, no algebraica.